## Coffee Shop Sales: Regression

### 1. Set up Spark and load data

In [15]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CoffeeShopRegression")
    .master("local[2]")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)

print("Spark Version:", spark.version)

Spark Version: 4.2.0


In [16]:
file_path = r"D:\Projects\Coffee Shop Sales - Cluster Computing\Dataset\Coffee Shop Sales.csv"
df = spark.read.csv(
    file_path,
    header = True,
    inferSchema = True
)
print("Total Rows:", df.count())
print("Total Columns:", len(df.columns))

df.show(5, truncate=False)

Total Rows: 149116
Total Columns: 11
+--------------+----------------+-------------------+---------------+--------+---------------+----------+----------+------------------+---------------------+---------------------------+
|transaction_id|transaction_date|transaction_time   |transaction_qty|store_id|store_location |product_id|unit_price|product_category  |product_type         |product_detail             |
+--------------+----------------+-------------------+---------------+--------+---------------+----------+----------+------------------+---------------------+---------------------------+
|1             |01-01-2023      |2026-10-20 07:06:11|2              |5       |Lower Manhattan|32        |3.0       |Coffee            |Gourmet brewed coffee|Ethiopia Rg                |
|2             |01-01-2023      |2026-10-20 07:08:56|2              |5       |Lower Manhattan|57        |3.1       |Tea               |Brewed Chai tea      |Spicy Eye Opener Chai Lg   |
|3             |01-01-2023      |

### 2. Prepare transaction and daily sales data

In [17]:
from pyspark.sql.functions import col, to_date, hour

# Correctly parse the transaction date
df = df.withColumn(
    "transaction_date",
    to_date(col("transaction_date"), "dd-MM-yyyy")
)

# Calculate revenue for each transaction
df = df.withColumn(
    "revenue",
    col("transaction_qty") * col("unit_price")
)

# Extract the transaction hour
df = df.withColumn(
    "hour",
    hour(col("transaction_time"))
)

print("Total rows:", df.count())
df.select(
    "transaction_date",
    "store_id",
    "transaction_qty",
    "unit_price",
    "revenue",
    "hour"
).show(5, truncate=False)

Total rows: 149116
+----------------+--------+---------------+----------+-------+----+
|transaction_date|store_id|transaction_qty|unit_price|revenue|hour|
+----------------+--------+---------------+----------+-------+----+
|2023-01-01      |5       |2              |3.0       |6.0    |7   |
|2023-01-01      |5       |2              |3.1       |6.2    |7   |
|2023-01-01      |5       |2              |4.5       |9.0    |7   |
|2023-01-01      |5       |1              |2.0       |2.0    |7   |
|2023-01-01      |5       |2              |3.1       |6.2    |7   |
+----------------+--------+---------------+----------+-------+----+
only showing top 5 rows


In [18]:
from pyspark.sql.functions import sum, count

daily_sales = (
    df.groupBy(
        "transaction_date",
        "store_id",
        "store_location"
    )
    .agg(
        sum("revenue").alias("daily_revenue"),
        count("*").alias("transaction_count")
    )
    .orderBy("transaction_date", "store_id")
)

print("Total daily records:", daily_sales.count())

daily_sales.show(10, truncate=False)

Total daily records: 543
+----------------+--------+---------------+-----------------+-----------------+
|transaction_date|store_id|store_location |daily_revenue    |transaction_count|
+----------------+--------+---------------+-----------------+-----------------+
|2023-01-01      |3       |Astoria        |868.4            |190              |
|2023-01-01      |5       |Lower Manhattan|788.3499999999999|176              |
|2023-01-01      |8       |Hell's Kitchen |851.4500000000002|184              |
|2023-01-02      |3       |Astoria        |925.5000000000001|212              |
|2023-01-02      |5       |Lower Manhattan|649.0500000000001|159              |
|2023-01-02      |8       |Hell's Kitchen |828.8000000000001|195              |
|2023-01-03      |3       |Astoria        |902.7500000000003|205              |
|2023-01-03      |5       |Lower Manhattan|756.0            |166              |
|2023-01-03      |8       |Hell's Kitchen |906.25           |211              |
|2023-01-04    

In [19]:
from pyspark.sql.functions import dayofweek, month, dayofmonth

daily_sales = (
    daily_sales
    .withColumn("day_of_week", dayofweek("transaction_date"))
    .withColumn("month", month("transaction_date"))
    .withColumn("day_of_month", dayofmonth("transaction_date"))
)

daily_sales.select(
    "transaction_date",
    "store_id",
    "day_of_week",
    "month",
    "day_of_month",
    "daily_revenue"
).show(10, truncate=False)

+----------------+--------+-----------+-----+------------+-----------------+
|transaction_date|store_id|day_of_week|month|day_of_month|daily_revenue    |
+----------------+--------+-----------+-----+------------+-----------------+
|2023-01-01      |3       |1          |1    |1           |868.4            |
|2023-01-01      |5       |1          |1    |1           |788.3499999999999|
|2023-01-01      |8       |1          |1    |1           |851.4500000000002|
|2023-01-02      |3       |2          |1    |2           |925.5000000000001|
|2023-01-02      |5       |2          |1    |2           |649.0500000000001|
|2023-01-02      |8       |2          |1    |2           |828.8000000000001|
|2023-01-03      |3       |3          |1    |3           |902.7500000000003|
|2023-01-03      |5       |3          |1    |3           |756.0            |
|2023-01-03      |8       |3          |1    |3           |906.25           |
|2023-01-04      |3       |4          |1    |4           |808.2500000000003|

### 3. Split data into training and testing sets

In [20]:
from pyspark.sql.functions import col, lit

train_data = daily_sales.filter(
    col("transaction_date") < lit("2023-06-01")
)

test_data = daily_sales.filter(
    col("transaction_date") >= lit("2023-06-01")
)

print("Training rows:", train_data.count())
print("Testing rows:", test_data.count())

Training rows: 453
Testing rows: 90


### 4. Train the linear regression model

In [21]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline

# Select input features
feature_cols = [
    "store_id",
    "day_of_week",
    "month",
    "day_of_month"
]

# Combine features into a single vector
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

# Create Linear Regression model
lr = LinearRegression(
    featuresCol="features",
    labelCol="daily_revenue",
    predictionCol="predicted_revenue"
)

# Create pipeline
pipeline = Pipeline(stages=[assembler, lr])

# Train the model
lr_model = pipeline.fit(train_data)

print("Linear Regression model trained successfully!")

Linear Regression model trained successfully!


### 5. Generate predictions

In [22]:
predictions = lr_model.transform(test_data)

predictions.select(
    "transaction_date",
    "store_id",
    "daily_revenue",
    "predicted_revenue"
).show(15, truncate=False)

+----------------+--------+------------------+------------------+
|transaction_date|store_id|daily_revenue     |predicted_revenue |
+----------------+--------+------------------+------------------+
|2023-06-01      |3       |1794.8499999999997|1725.675306150106 |
|2023-06-01      |5       |1602.9500000000005|1732.8888160838794|
|2023-06-01      |8       |1829.2000000000003|1743.7090809845397|
|2023-06-02      |3       |1915.4500000000003|1726.2965821850005|
|2023-06-02      |5       |1439.4499999999998|1733.5100921187739|
|2023-06-02      |8       |1701.6000000000008|1744.3303570194341|
|2023-06-03      |3       |1953.5000000000007|1726.9178582198947|
|2023-06-03      |5       |1600.3500000000006|1734.131368153668 |
|2023-06-03      |8       |1612.8000000000006|1744.9516330543283|
|2023-06-04      |3       |1760.9            |1743.6250688659436|
|2023-06-04      |5       |1595.3500000000001|1750.838578799717 |
|2023-06-04      |8       |1628.9000000000008|1761.6588437003772|
|2023-06-0

### 6. Evaluate model performance

In [23]:
from pyspark.ml.evaluation import RegressionEvaluator

metrics = ["r2", "rmse", "mae"]

for metric in metrics:
    evaluator = RegressionEvaluator(
        labelCol="daily_revenue",
        predictionCol="predicted_revenue",
        metricName=metric
    )

    score = evaluator.evaluate(predictions)
    print(f"{metric.upper()}: {score:.4f}")

R2: -0.0409
RMSE: 294.0383
MAE: 231.5454


In [24]:
print("Linear Regression Model Evaluation")
print("----------------------------------")

for metric in ["r2", "rmse", "mae"]:
    evaluator = RegressionEvaluator(
        labelCol="daily_revenue",
        predictionCol="predicted_revenue",
        metricName=metric
    )
    print(f"{metric.upper()}: {evaluator.evaluate(predictions):.4f}")

Linear Regression Model Evaluation
----------------------------------
R2: -0.0409
RMSE: 294.0383
MAE: 231.5454
